# scFoundation Cross-cell C2 · 编码器 token 配对消融

两臂沿用冻结的官方预训练权重、同一 GEARS 解码器、相同训练和验证行及对照、300 个拟合 microbatch、60 次优化器更新和同一评估协议。官方臂只把非零表达基因送入 MAE 编码器。全允许基因臂加入 3,194 个评分轴上表达为零的基因 token。两臂均保留官方完整解码器输入。测试留出表达仍为零，且不会作为新增编码器 token。这个干预估计零表达 token 进入 MAE 编码器的影响，结果限同一模型和单个训练种子。

在服务器运行。当前目录的 `token-ablations.json` 键 `released_nonzero` 和 `all_allowed` 指向两个正式结果目录。响应以同条件对照均值为基准。表征取 300 个控制输入的 GEARS cross_gene_state 均值。有效秩使用中心化奇异值平方能量，除以可达最大秩。余弦几何用配对扰动间的上三角比较。

主终点为 TriShift Δ Pearson。两臂差值按同一扰动做 10,000 次配对 bootstrap，区间反映固定权重在当前扰动集合下的不确定性。

In [ ]:

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

root = Path.cwd()
runs = json.loads((root / "token-ablations.json").read_text())
arms = ["released_nonzero", "all_allowed"]
labels = {"released_nonzero": "Released nonzero", "all_allowed": "All eligible"}
identity_fields = (
    "source_commit", "official_commit", "weights_sha256",
    "cohort_sha256", "shared_fit_support_file_sha256", "paired_rows_sha256",
    "test_panel_gene_order_sha256", "score_panel_sha256", "fit_selection_sha256", "val_selection_sha256",
)
identities, condition_order, gene_order, true_delta = set(), None, None, None
summary, by_condition, analysis = {}, {}, {}
for arm in arms:
    path = root / runs[arm]
    r = json.loads((path / "result.json").read_text())
    assert r["encoder_tokens"] == arm and r["status"] == "evaluated"
    assert (r["fit_microbatches"], r["optimizer_updates"], r["test_updates"], r["test_panel_genes"]) == (300, 60, 0, 3194)
    identities.add(tuple(r[k] for k in identity_fields))
    vector_path = path / "condition-representations.npz"
    assert hashlib.sha256(vector_path.read_bytes()).hexdigest() == r["diagnostic_sha256"]
    with np.load(vector_path, allow_pickle=False) as vectors:
        z = vectors["cross_gene_state"].astype(float)
        prediction = vectors["predicted_delta"]
        truth = vectors["truth_delta"]
        conditions, genes = vectors["conditions"], vectors["genes"]
    assert len(set(conditions)) == z.shape[0] == prediction.shape[0] == 128
    if condition_order is not None:
        np.testing.assert_array_equal(conditions, condition_order)
        np.testing.assert_array_equal(genes, gene_order)
        np.testing.assert_allclose(truth, true_delta, rtol=1e-7, atol=1e-10)
    condition_order, gene_order, true_delta = conditions, genes, truth
    zc = z - z.mean(0)
    denominator = np.linalg.norm(zc, axis=1, keepdims=True)
    z_unit = np.divide(zc, denominator, out=np.zeros_like(zc), where=denominator > 0)
    truth_norm = np.linalg.norm(truth, axis=1)
    truth_unit = np.divide(truth, truth_norm[:, None], out=np.zeros_like(truth), where=truth_norm[:, None] > 0)
    triangle = np.triu_indices(len(zc), k=1)
    z_angles = (z_unit @ z_unit.T)[triangle]
    truth_angles = (truth_unit @ truth_unit.T)[triangle]
    singular = np.linalg.svd(zc, compute_uv=False)
    spectrum = singular ** 2
    p = spectrum / spectrum.sum()
    nonzero = p[p > 0]
    rank = np.exp(-np.sum(nonzero * np.log(nonzero)))
    pred_norm = np.linalg.norm(prediction, axis=1)
    amplitude = np.divide(pred_norm, truth_norm, out=np.full_like(pred_norm, np.nan), where=truth_norm > 0)
    cosine = np.divide((prediction * truth).sum(1), pred_norm * truth_norm,
                       out=np.full_like(pred_norm, np.nan), where=(pred_norm * truth_norm) > 0)
    square = (prediction - truth) ** 2
    recorded = [json.loads(line) for line in (path / "conditions.jsonl").read_text().splitlines()]
    assert list(conditions) == [row["condition_id"] for row in recorded]
    metrics = [{**{m["metric_id"]:m["value"] for m in row["results"]},
                "mse": row["mean_mse"], "retrieval": row["retrieval"]} for row in recorded]
    table = pd.DataFrame(metrics, index=pd.Index(conditions, name="condition"))
    table["amplitude_ratio"], table["response_cosine"] = amplitude, cosine
    table["response_mse"] = square.mean(1)
    by_condition[arm] = table
    summary[arm] = {
        "effective_rank": rank,
        "normalized_rank": rank / min(z.shape[0] - 1, z.shape[1]),
        "relative_centered_variance": (zc ** 2).sum() / (z ** 2).sum(),
        "median_pair_cosine": np.median(z_angles),
        "representation_truth_geometry_spearman": spearmanr(z_angles, truth_angles).statistic,
        **dict(table.mean()),
    }
    analysis[arm] = (np.cumsum(p), z_angles, truth_angles, square.mean(0))
assert len(identities) == 1
summarized = pd.DataFrame(summary).T
summarized.to_csv(root / "ablation-metrics.csv")
pd.concat(by_condition, names=["arm"]).to_csv(root / "ablation-condition-metrics.csv")
display(summarized.round(5))

fig, axes = plt.subplots(2, 3, figsize=(14, 7.3), layout="constrained")
for arm in arms:
    curve, angles, truth_angles, by_gene = analysis[arm]
    label = labels[arm]
    axes[0, 0].plot(np.arange(1, len(curve) + 1), curve, label=label)
    axes[0, 1].hist(angles, bins=24, histtype="step", density=True, label=label)
    axes[0, 2].plot(truth_angles, angles, ".", markersize=1.8, alpha=.13, label=label)
axes[0, 0].set(title="Centered spectrum", xlabel="Ordered singular component", ylabel="Cumulative energy")
axes[0, 1].set(title="Latent pair cosine", xlabel="Centered cosine")
axes[0, 2].set(title="Latent vs observed response geometry", xlabel="True response pair cosine", ylabel="Latent pair cosine")
for ax, key, title in zip(axes[1], ["amplitude_ratio","response_cosine","response_mse"],
                          ["Response norm ratio", "Response direction cosine", "Response mean MSE"]):
    ax.boxplot([by_condition[a][key].dropna() for a in arms], tick_labels=[labels[a] for a in arms])
    ax.set_title(title)
for ax in axes[0]:
    ax.legend()
fig.savefig(root / "token-response-representations.png", dpi=170)
plt.show()

cols = list(summarized.columns)
nrows = (len(cols) + 3) // 4
fig, axes = plt.subplots(nrows, 4, figsize=(13, 3.1 * nrows), layout="constrained")
for ax, key in zip(axes.flat, cols):
    bars = ax.bar([labels[a] for a in arms], summarized[key], color=["#285d9f", "#c66d24"])
    ax.bar_label(bars, fmt="%.4g")
    ax.set_title(key.replace("_", " "), fontsize=9)
    ax.tick_params(axis="x", labelrotation=24, labelsize=8)
for ax in list(axes.flat)[len(cols):]:
    ax.axis("off")
fig.savefig(root / "token-ablation-metrics.png", dpi=170)
plt.show()

gene_error = pd.DataFrame({labels[arm]: analysis[arm][3] for arm in arms}, index=gene_order)
gene_error.index.name = "gene"
worst_genes = gene_error.assign(mean_error=gene_error.mean(axis=1)).nlargest(20, "mean_error")
worst_genes.to_csv(root / "token-highest-error-genes.csv")
fig, ax = plt.subplots(figsize=(9, 7), layout="constrained")
worst_genes[list(labels.values())].iloc[::-1].plot.barh(ax=ax, color=["#285d9f", "#c66d24"])
ax.set(xlabel="Mean squared response error across conditions", ylabel="Gene")
fig.savefig(root / "token-highest-error-genes.png", dpi=170)
plt.show()

paired_metrics = [
    "trishift_pearson_delta_all", "txpert_macro_pearson_delta_all", "systema_pearson_all",
    "mse", "retrieval", "amplitude_ratio", "response_cosine", "response_mse",
]
rng = np.random.default_rng(20261125)
rows = []
for metric in paired_metrics:
    pair = (by_condition["all_allowed"][metric] - by_condition["released_nonzero"][metric]).dropna()
    if len(pair) < 2:
        raise ValueError(f"insufficient finite paired conditions: {metric}")
    delta = pair.to_numpy()
    draws = rng.integers(len(delta), size=(10000, len(delta)))
    lower, upper = np.quantile(delta[draws].mean(1), [.025, .975])
    rows.append(dict(metric=metric, pairs=len(delta), delta=delta.mean(), low=lower, high=upper))
comparison = pd.DataFrame(rows)
comparison.to_csv(root / "token-paired-effects.csv", index=False)
display(comparison.round(5))
fig, axes = plt.subplots(2, 4, figsize=(14, 5.6), layout="constrained")
for ax, r in zip(axes.flat, comparison.itertuples(), strict=True):
    ax.errorbar(r.delta, 0, xerr=[[r.delta-r.low], [r.high-r.delta]], fmt="o", color="#285d9f")
    ax.axvline(0, lw=1, color=".6")
    ax.set(title=r.metric.replace("_", " "), yticks=[])
fig.suptitle("All eligible − released nonzero · paired 95% bootstrap intervals")
fig.savefig(root / "token-paired-effects.png", dpi=170)
plt.show()
